##**How much worse does Llama-2-7B perform when adding two 7-digit numbers instead of two 1-digit numbers?**

In [1]:
import re
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from time import sleep
from sklearn.metrics import mean_absolute_error
from huggingface_hub import notebook_login

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


In [2]:
# Helper function for debugging
def dprint(s, debug):
    if debug:
        print(s)

In [3]:
# --- Check Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [4]:
# --- Model Loading ---
# We'll run our models using Hugging Face's transformers library on HPC/ Google Colab/ Lightning.ai
# The Llama models are gated, meaning you must request access on their Hugging Face pages.
# Once you have access, you need to log in here to download the model weights.

# Run this command in your terminal when you are running this notebook for the 1st time
# git config --global credential.helper store

print("Please log in to your Hugging Face account.")
notebook_login()

Please log in to your Hugging Face account.


In [5]:
# This will be our primary model for most of the assignment.
model_id_1 = "meta-llama/Llama-2-7b-chat-hf"

In [6]:
print(f"\nLoading tokenizer for {model_id_1}...")
# The tokenizer turns our text prompt into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_id_1)

print(f"Loading model: {model_id_1}...")
# This downloads the model weights to your environment.
# torch_dtype=torch.bfloat16 uses half-precision floats to save memory.
# device_map="auto" automatically puts the model on the GPU if available.
model = AutoModelForCausalLM.from_pretrained(
    model_id_1,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_1} model loaded successfully!")


Loading tokenizer for meta-llama/Llama-2-7b-chat-hf...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

meta-llama/Llama-2-7b-chat-hf model loaded successfully!


In [ ]:
!pip -q install transformers accelerate huggingface_hub pandas scikit-learn

In [7]:
def call_model(prompt, student_configs, post_processing_fn, model_obj, tokenizer_obj, debug=False):
    """
    Generates a response using the provided local Hugging Face model and tokenizer.
    """
    # 1. Tokenize the input prompt
    inputs = tokenizer_obj(prompt, return_tensors="pt").to(device)

    hf_configs = student_configs.copy()
    if 'max_tokens' in hf_configs:
        # `generate` uses `max_new_tokens` to specify the length of the output
        hf_configs['max_new_tokens'] = hf_configs.pop('max_tokens')
    if 'stop' in hf_configs:
        del hf_configs['stop'] # Stop sequences are handled differently; we'll ignore for simplicity

    # 2. Generate output tokens
    outputs = model_obj.generate(**inputs, **hf_configs).to(device)

    # 3. Decode the generated tokens back to a string
    # We slice the output to only get the newly generated text, not the original prompt
    result_new = tokenizer_obj.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    dprint("************ Prompt ************", debug)
    dprint(prompt, debug)
    dprint("\n************ Raw Response ************", debug)
    dprint(result_new, debug)

    # 4. Apply post-processing to extract the final answer
    final_output = post_processing_fn(result_new)

    dprint("\n************ Final Output ************", debug)
    dprint(final_output, debug)

    return final_output

In [8]:
def get_addition_pairs(lower_bound, upper_bound, rng):
    """Generates two random integers within a specified range."""
    int_a = int(np.ceil(rng.uniform(lower_bound, upper_bound)))
    int_b = int(np.ceil(rng.uniform(lower_bound, upper_bound)))
    return int_a, int_b

def test_range(added_prompt, prompt_configs, rng, model_obj, tokenizer_obj, n_sample=30,
               lower_bound=1, upper_bound=10, fixed_pairs=None,
               pre_processing=lambda x:x, post_processing=lambda y:y,
               debug=False):
    """
    Tests a language model's addition performance over a range of numbers.

    Args:
        added_prompt (tuple): A tuple containing the prefix and suffix for the prompt.
        prompt_configs (dict): Configuration parameters for the model's generate function.
        rng (numpy.random.Generator): A random number generator instance.
        model_obj (transformers.PreTrainedModel): The loaded Hugging Face model object.
        tokenizer_obj (transformers.PreTrainedTokenizer): The loaded Hugging Face tokenizer object.
        n_sample (int): The number of random pairs to generate if fixed_pairs is None.
        lower_bound (int): The lower bound for number generation.
        upper_bound (int): The upper bound for number generation.
        fixed_pairs (list, optional): A list of specific integer tuples to test.
        pre_processing (function): A function to apply to the input string before prompting.
        post_processing (function): A function to extract the integer answer from the model's output.
        debug (bool): If True, prints detailed debugging information.

    Returns:
        dict: A dictionary containing performance metrics (res, acc, mae, prompt_length).
    """
    # --- Lists for storing results ---
    int_as = []
    int_bs = []
    answers = []
    model_responses = []
    correct = []
    prompts = []

    # --- Determine the test cases ---
    iterations = range(n_sample) if fixed_pairs is None else fixed_pairs

    for v in tqdm(iterations):
        if fixed_pairs is None:
            # Generate two new numbers if no fixed pairs are provided
            int_a, int_b = get_addition_pairs(lower_bound=lower_bound, upper_bound=upper_bound, rng=rng)
        else:
            # Use the provided fixed pairs
            int_a, int_b = v

        # --- Construct the prompt for two numbers ---
        fixed_prompt = f'{int_a}+{int_b}'
        fixed_prompt = pre_processing(fixed_prompt)

        prefix, suffix = added_prompt
        prompt = prefix + fixed_prompt + suffix

        # --- Get the model's response ---
        model_response = call_model(prompt, prompt_configs, post_processing, model_obj, tokenizer_obj, debug=debug)

        # --- Calculate the correct answer for two numbers ---
        answer = int_a + int_b

        # --- Append all results for analysis ---
        int_as.append(int_a)
        int_bs.append(int_b)
        prompts.append(prompt)
        answers.append(answer)
        model_responses.append(model_response)
        correct.append((answer == model_response))
        sleep(0.1)

    # --- Create a DataFrame to display the results for two numbers ---
    df = pd.DataFrame({
        'int_a': int_as,
        'int_b': int_bs,
        'prompt': prompts,
        'answer': answers,
        'response': model_responses,
        'correct': correct
    })
    print(df)

    # --- Calculate and return performance metrics ---
    mae = mean_absolute_error(df['answer'], df['response'])
    acc = df.correct.sum() / len(df)
    prompt_length = len(prefix) + len(suffix)
    res = acc * (1 / prompt_length) * (1 - mae / (1 * 10**4))

    return {'res': res, 'acc': acc, 'mae': mae, 'prompt_length': prompt_length}

###  Part 1. Zero Shot Addition

**Example: Zero-shot single-digit addition**

In [9]:
# All of this remains the same
added_prompt = ('Question: What is ', '?\\nAnswer: ')
prompt_config = {'max_tokens': 8,
                'temperature': 0.7,
                'top_k': 50,
                'top_p': 0.6,
                'repetition_penalty': 1,
                'stop': []}

def your_pre_processing(input_string):
    return input_string

def your_post_processing(output_string):
    only_digits = re.sub(r"\D", "", output_string)
    try:
        res = int(only_digits)
    except:
        res = 0
    return res

# The model name string is no longer passed to the function
# It was used in the previous cell to load the 'model' and 'tokenizer' objects
print(f"Testing model: {model_id_1}")
seed = 0
rng = np.random.default_rng(seed)

# This is the only line that changes
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model,
    tokenizer_obj=tokenizer,
    n_sample=10,
    lower_bound=1,
    upper_bound=10,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res)

Testing model: meta-llama/Llama-2-7b-chat-hf


  0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

   int_a  int_b                             prompt  answer  response  correct
0      7      4   Question: What is 7+4?\nAnswer:       11        11     True
1      2      2   Question: What is 2+2?\nAnswer:        4         4     True
2      9     10  Question: What is 9+10?\nAnswer:       19        19     True
3      7      8   Question: What is 7+8?\nAnswer:       15        15     True
4      6     10  Question: What is 6+10?\nAnswer:       16        16     True
5      9      2   Question: What is 9+2?\nAnswer:       11        11     True
6      9      2   Question: What is 9+2?\nAnswer:       11        11     True
7      8      3   Question: What is 8+3?\nAnswer:       11        11     True
8      9      6   Question: What is 9+6?\nAnswer:       15        15     True
9      4      5   Question: What is 4+5?\nAnswer:        9         9     True
{'res': np.float64(0.034482758620689655), 'acc': np.float64(1.0), 'mae': 0.0, 'prompt_length': 29}


**Example: Zero-shot 7-digit addition**

In [10]:

prompt_config['max_tokens'] = 8
rng = np.random.default_rng(seed)

# The call to test_range is updated to pass the model and tokenizer objects.
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10,
    lower_bound=1000000,
    upper_bound=9999999,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)

print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10154936    False  
1   2517511     True  
2  17535200    False  
3   6459722    False  
4   5892625    

-----------




**Q3a.** In your opinion, what are some factors that cause language model performance to deteriorate from 1 digit to 7 digits?

**Answer:**

In the result of the above experiment, Llama-2-7B answered all single digit questions correctly, but it only got one correct answer for 7-digit numbers. A language model generates tokens based on the dataset and pattern rather than calculating numbers arithmetically. Because of this, the model may struggle to keep track of information from earlier calculation steps and carry values throughout the calculation.

-----------

**Q3b**. Play around with the config parameters ('max_tokens','temperature','top_k','top_p','repetition_penalty')
* What does each parameter represent?
* How does increasing each parameter change the generation?

In [11]:
experiments = {
    'max_tokens': [8, 20, 50],
    'temperature': [0.3, 0.7, 1.2],
    'top_k': [10, 50, 100],
    'top_p': [0.3, 0.6, 0.95],
    'repetition_penalty': [1.0, 1.1, 1.3]
}

results_q3b = {}

original_sampling = model.generation_config.do_sample

try:
    for parameter, values in experiments.items():

        model.generation_config.do_sample = (parameter != 'max_tokens')

        for value in values:

            experiment_config = prompt_config.copy()
            experiment_config[parameter] = value

            print(f"\nTesting {parameter} = {value}")

            torch.manual_seed(0)
            rng = np.random.default_rng(seed)

            res = test_range(
                added_prompt=added_prompt,
                prompt_configs=experiment_config,
                rng=rng,
                model_obj=model,
                tokenizer_obj=tokenizer,
                n_sample=10,
                lower_bound=1000000,
                upper_bound=9999999,
                fixed_pairs=None,
                pre_processing=your_pre_processing,
                post_processing=your_post_processing,
                debug=False
            )

            results_q3b[(parameter, value)] = res

            print(res)

finally:
    model.generation_config.do_sample = original_sampling


Testing max_tokens = 8


  0%|          | 0/10 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10154736    False  
1   2517511     True  
2  17535200    False  
3  13625188    False  
4   5892625    

  0%|          | 0/10 [00:00<?, ?it/s]

...

[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10154936    False  
1   2517501    False  
2  17535200    False  
3  13625088    False  
4   5892625    

  0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10159436    False  
1      1368    False  
2   7657648    False  
3  13805198    False  
4   5703896    

In [12]:
for (parameter, value), result in results_q3b.items():
    print(
        parameter, value,
        "Accuracy:", result['acc'],
    )

max_tokens 8 Accuracy: 0.2
max_tokens 20 Accuracy: 0.2
max_tokens 50 Accuracy: 0.0
temperature 0.3 Accuracy: 0.2
temperature 0.7 Accuracy: 0.2
temperature 1.2 Accuracy: 0.2
top_k 10 Accuracy: 0.2
top_k 50 Accuracy: 0.2
top_k 100 Accuracy: 0.2
top_p 0.3 Accuracy: 0.2
top_p 0.6 Accuracy: 0.2
top_p 0.95 Accuracy: 0.2
repetition_penalty 1.0 Accuracy: 0.2
repetition_penalty 1.1 Accuracy: 0.1
repetition_penalty 1.3 Accuracy: 0.0


**Answer:**

I experimented with five generation parameters using the same ten 7-digit addition problems.

1. Max Tokens: This determines the maximum number of tokens the model can generate. When I increased it from 8 to 20 and 50, accuracy changed from 20% to 20% and then 0%. I noticed that allowing more tokens caused the model to continue generating additional text and numbers instead of stopping after the answer. This also made it harder for the postprocessing function to extract the correct result.

2. Temperature: Temperature controls the randomness of token generation when sampling is enabled. Increasing it generally makes the outputs more diverse and less predictable. In my experiment, accuracy stayed at 20% at 0.3, 0.7, and 1.2. I think increasing randomness is not necessarily helpful for arithmetic problems because there is only one correct answer.

3. Top K: Top K limits the model to selecting from the K most probable next tokens. Increasing it allows more possible tokens to be considered during sampling. However, I noticed that increasing Top K from 10 to 100 did not change the generated answers or accuracy, which remained at 20%.

4. Top P: Top P controls which tokens can be sampled based on their cumulative probability. Increasing it allows a wider range of possible tokens. In my experiment, accuracy was 20% at 0.3, 0.6, and 0.95. Although the outputs changed, increasing Top P did not consistently improve accuracy.

5. Repetition Penalty: This parameter discourages the model from repeating previously used tokens. Increasing it generally reduces repetition, but repeated digits can be necessary in arithmetic. In my experiment, accuracy decreased from 20% at 1.0 to 10% at 1.1 and 0% at 1.3, suggesting that increasing the penalty did not help the model produce more correct answers.

Overall, I noticed that changing generation parameters did not consistently improve arithmetic accuracy. Increasing the number of generated tokens also created problems with extracting the final answer, while parameters controlling randomness did not show a clear benefit for mathematical calculations.

Q3c. Do 7-digit addition with Qwen3 8B.

* How does the performance change?
* What are some factors that cause this change?

In [11]:
# --- Before loading Qwen 3, offload Llama 2 to free up VRAM ---

# 1. Delete the model and tokenizer variables from memory.
# Replace 'model' and 'tokenizer' with the actual variable names you used for Llama 2.
del model
del tokenizer

# 2. Run Python's garbage collector and empty PyTorch's CUDA cache.
# This is the crucial step to actually release the GPU memory.
gc.collect()
torch.cuda.empty_cache()

print("Llama 2 model offloaded and GPU memory cleared.")

Llama 2 model offloaded and GPU memory cleared.


In [12]:
# --- Load Qwen 3 8B ---
# This is a different model, so we need to load its specific tokenizer and weights.
model_id_2 = "Qwen/Qwen3-8B"

print(f"\nLoading tokenizer for {model_id_2}...")
tokenizer_2 = AutoTokenizer.from_pretrained(model_id_2)

print(f"Loading model: {model_id_2}...")
model_2 = AutoModelForCausalLM.from_pretrained(
    model_id_2,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_2} model loaded successfully!")


Loading tokenizer for Qwen/Qwen3-8B...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen/Qwen3-8B model loaded successfully!


In [13]:
# --- Test on 7-digit addition ---
prompt_config['max_tokens'] = 8

print(prompt_config)
rng = np.random.default_rng(seed)
res_2 = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model_2,              # Pass the loaded model object
    tokenizer_obj=tokenizer_2,      # Pass the loaded tokenizer object
    n_sample=10,
    lower_bound=1000000,
    upper_bound=9999999,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res_2)

{'max_tokens': 8, 'temperature': 0.7, 'top_k': 50, 'top_p': 0.6, 'repetition_penalty': 1, 'stop': []}


  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10160736     True  
1   2517511     True  
2  17534232     True  
3  14025191     True  
4  15308276    

-----------

**Answer:**

Performance has greatly improved when it comes to implementing Qwen3 compared to Llama-2. I think the improvement may be partly explained by differences in the models' post-training methods. According to Touvron et al. (2023), Llama-2-Chat uses supervised fine-tuning (SFT) and reinforcement learning from human feedback (RLHF), with an emphasis on helpfulness and safety. In contrast, the Qwen3 technical report (Qwen Team, 2025) describes reasoning-focused reinforcement learning with verifiable rewards for its larger models. Since mathematical problems have objectively verifiable answers, I think this training approach may contribute to Qwen3's stronger performance on 7-digit addition. However, the experiment does not prove that the training methods alone caused the difference.

-----------

**Q3d.** Here we're giving our language model the prior that the sum of two 7-digit numbers must have a maximum of 8 digits. (by setting max_token=8). What if we remove this prior by increasing the max_token to 20?
* Does the model perform well?
* What are some reasons why?

**Answer:**

After increasing max_tokens from 8 to 20, Qwen3-8B's accuracy decreased, specifically getting right answers from 9 out of 10 to 5 out of 10. (90%->50% accuracy) Looking at the generated results, I noticed that the model sometimes produced the correct answer initially but continued generating additional numbers instead of stopping. Since the postprocessing function combines all digits from the generated output, even an initially correct answer could be evaluated as incorrect.

I think limiting the output length to 8 tokens helped the model produce shorter answers, while allowing 20 tokens gave it more opportunities to generate unnecessary text. Therefore, the decrease in measured accuracy does not necessarily mean the model's arithmetic ability became worse, but rather that longer generation makes it more difficult to extract the correct answer.

In [14]:

prompt_config['max_tokens'] = 20
rng = np.random.default_rng(seed)
res_2 = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model_2,              # Pass the loaded model object
    tokenizer_obj=tokenizer_2,      # Pass the loaded tokenizer object
    n_sample=10,
    lower_bound=1000000,
    upper_bound=9999999,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res_2)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

         response  correct  
0        10160736     True  
1         2517511     True  
2  17534232831943    False  
3  14025191783

In [15]:
# 1. Delete the model and tokenizer variables from memory.
# Replace 'model' and 'tokenizer' with the actual variable names you used for Llama 2.
del model_2
del tokenizer_2

# 2. Run Python's garbage collector and empty PyTorch's CUDA cache.
# This is the crucial step to actually release the GPU memory.
gc.collect()
torch.cuda.empty_cache()

print(f"{model_id_2} offloaded and GPU memory cleared.")

Qwen/Qwen3-8B offloaded and GPU memory cleared.


### Part 2. In Context Learning

We will try to improve the performance of 7-digit addition via in-context learning.
We will use [llama-2-7b]. Below is a simple example.

In [16]:
print(f"\nLoading tokenizer for {model_id_1}...")
# The tokenizer turns our text prompt into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_id_1)

print(f"Loading model: {model_id_1}...")
# This downloads the model weights to your environment.
# torch_dtype=torch.bfloat16 uses half-precision floats to save memory.
# device_map="auto" automatically puts the model on the GPU if available.
model = AutoModelForCausalLM.from_pretrained(
    model_id_1,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_1} model loaded successfully!")


Loading tokenizer for meta-llama/Llama-2-7b-chat-hf...
Loading model: meta-llama/Llama-2-7b-chat-hf...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

meta-llama/Llama-2-7b-chat-hf model loaded successfully!


In [17]:

added_prompt = ('Question: What is 3+7?\nAnswer: 10\n Question: What is ', '?\nAnswer: ') # Question: What is a+b?\nAnswer:
prompt_config = {'max_tokens': 8,
                'temperature': 0.7,
                'top_k': 50,
                'top_p': 0.6,
                'repetition_penalty': 1,
                'stop': []}
rng = np.random.default_rng(seed)
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10,
    lower_bound=1000000,
    upper_bound=9999999,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)

print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

     int_a    int_b                                             prompt  \
0  6732655  3428081  Question: What is 3+7?\nAnswer: 10\n Question:...   
1  1368762  1148749  Question: What is 3+7?\nAnswer: 10\n Question:...   
2  8319432  9214800  Question: What is 3+7?\nAnswer: 10\n Question:...   
3  6459722  7565469  Question: What is 3+7?\nAnswer: 10\n Question:...   
4  5892625  9415651  Question: What is 3+7?\nAnswer: 10\n Question:...   
5  8342682  1024647  Question: What is 3+7?\nAnswer: 10\n Question:...   
6  8716638  1302271  Question: What is 3+7?\nAnswer: 10\n Question:...   
7  7566899  2580901  Question: What is 3+7?\nAnswer: 10\n Question:...   
8  8768610  5873151  Question: What is 3+7?\nAnswer: 10\n Question:...   
9  3697407  4804185  Question: What is 3+7?\nAnswer: 10\n Question:...   

     answer  response  correct  
0  10160736  10150436    False  
1   2517511   2517511     True  
2  17534232  17533282    False  
3  14025191   7322295    False  
4  15308276  1030816

**Q4a**.
* How does the performance change with the baseline in-context learning prompt? (compare with "Example: Zero-shot 7-digit addition" in Q1)
* What are some factors that cause this change?


**Answer:**

Compared to the zero-shot experiment, the accuracy stayed at 10%, getting 1 answer correct out of 10 in both cases, while the MAE decreased from 2,032,015.8 to 1,261,705.6. I think one possible reason is that the example is too simple compared with the actual task. Although it shows the model what kind of answer to generate, it doesn't explain how to handle multiple digits or carry values across different positions.

I am guessing another reason is that Llama-2-Chat uses SFT and RLHF to improve helpfulness and alignment with human preferences according to Touvron et al. (2023). In contrast, research on RL with verifiable rewards, such as DeepSeek-R1, shows how models can receive direct feedback based on mathematical correctness. That being said, Llama's training objectives may have some limitations for arithmetic calculations.

-----------

Now we will remove the prior on output length and re-evaluate the performance of our baseline one-shot learning prompt. We need to modify our post processing function to extract the answer from the output sequence. In this case, it is the number in the first line that starts with "Answer: ".

**Q4b**.
* Modify the post processing function
* How does the performance change when we relax the output length constraint? (compare with Q4a)
* What are some factors that cause this change?

**Answer:**

I modified the postprocessing function because the original function uses re.sub(r"\D", "", output_string) to remove every character that isn't a number and combined every number in the generated response into a single integer. Instead, I used split("\n", 1)[0] to extract the first generated line and re.match() to identify its numerical answer. This allows the model to generate additional text without including unrelated numbers in the evaluated answer.

Next, I increased max_tokens from 8 to 50 to relax the output length constraint, however the accuracy decreased from 10% to 0%. It didn't get any correct answers. I noticed that some predictions were close to correct answers but still output incorrect digits, which means that the model continued to struggle with multi-digit arithmetic.

Although the new postprocessing function helps extract the intended answer reliably, it can't help correct the arithmetic errors produced by the model. On top of this, increasing the output limit doesn't necessarily improve its ability to calculate mathematically either. Overall, the factor of this limitation would be the model's existing ability to perform calculation.

In [18]:
#Write your updated post processing function here

def your_post_processing(output_string):
    first_line = output_string.split("\n", 1)[0].strip()

    match = re.match(r"^(?:Answer:\s*)?(\d+)\b", first_line)

    if match:
        return int(match.group(1))

    return 0


In [19]:
sample_output = """10150436
Question: What is 1234567+2345678?
Answer: 3580245"""

print(your_post_processing(sample_output))

10150436


In [20]:
prompt_config['max_tokens'] = 50 # changed from 8, assuming we don't know the output length

rng = np.random.default_rng(seed)
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10,
    lower_bound=1000000,
    upper_bound=9999999,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

     int_a    int_b                                             prompt  \
0  6732655  3428081  Question: What is 3+7?\nAnswer: 10\n Question:...   
1  1368762  1148749  Question: What is 3+7?\nAnswer: 10\n Question:...   
2  8319432  9214800  Question: What is 3+7?\nAnswer: 10\n Question:...   
3  6459722  7565469  Question: What is 3+7?\nAnswer: 10\n Question:...   
4  5892625  9415651  Question: What is 3+7?\nAnswer: 10\n Question:...   
5  8342682  1024647  Question: What is 3+7?\nAnswer: 10\n Question:...   
6  8716638  1302271  Question: What is 3+7?\nAnswer: 10\n Question:...   
7  7566899  2580901  Question: What is 3+7?\nAnswer: 10\n Question:...   
8  8768610  5873151  Question: What is 3+7?\nAnswer: 10\n Question:...   
9  3697407  4804185  Question: What is 3+7?\nAnswer: 10\n Question:...   

     answer   response  correct  
0  10160736  101504423    False  
1   2517511    2517601    False  
2  17534232   17533242    False  
3  14025191    7324381    False  
4  15308276   1

-----------

**Q4c.** Let's change our one-shot learning example to something more "in-distribution". Previously we were using 1-digit addition as an example. Let's change it to 7-digit addition (1234567+1234567=2469134).
* Evaluate the performance with max_tokens = 8.
* Evaluate the performance with max_tokens = 50.
* How does the performance change from 1-digit example to 7-digit example?

**Answer:**

max_tokens = 8 : res = -0.460, acc = 0.2, mae = 1,826,568.9, prompt_length = 79

max_tokens = 50 : res = 0.0, acc = 0.0, mae = 826,043.6, prompt_length = 79

When I used max_tokens=8, accuracy improved from 10% with the one-digit example to 20% with the seven-digit example. I think this improvement may be related to the example being more in-distribution, since the model sees a problem with a similar number of digits and a similar answer format.

However, when I increased max_tokens to 50, the recorded accuracy dropped to 0%. I noticed that the responses contained unusually long sequences of digits, suggesting that the model continued generating additional text or numbers, which may not have been extracted correctly by the postprocessing function.

Overall, I think providing a more relevant example can not guarantee accurate mathematical reasoning. As we can see in the cell down below, the model may still struggle to keep track of digit positions and carry values.

In [21]:

prompt_config['max_tokens'] = 8
added_prompt = ('Question: What is 1234567+1234567?\nAnswer: 2469134\nQuestion: What is ', '?\nAnswer: ') # Question: What is a+b?\nAnswer:
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10,
    lower_bound=1000000,
    upper_bound=9999999,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

     int_a    int_b                                             prompt  \
0  1254878  2118550  Question: What is 1234567+1234567?\nAnswer: 24...   
1  7035620  6824705  Question: What is 1234567+1234567?\nAnswer: 24...   
2  6538466  4453098  Question: What is 1234567+1234567?\nAnswer: 24...   
3  9974889  9827518  Question: What is 1234567+1234567?\nAnswer: 24...   
4  7169878  6854133  Question: What is 1234567+1234567?\nAnswer: 24...   
5  7196020  4500293  Question: What is 1234567+1234567?\nAnswer: 24...   
6  2215869  7493395  Question: What is 1234567+1234567?\nAnswer: 24...   
7  5728189  3792177  Question: What is 1234567+1234567?\nAnswer: 24...   
8  5372518  9005390  Question: What is 1234567+1234567?\nAnswer: 24...   
9  9406391  4220157  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0   3373428   3373428     True  
1  13860325  13851925    False  
2  10991564  10992164    False  
3  19802407  19802497    False  
4  14024011  1396401

In [22]:

prompt_config['max_tokens'] = 50
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10,
    lower_bound=1000000,
    upper_bound=9999999,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

     int_a    int_b                                             prompt  \
0  6143768  3896825  Question: What is 1234567+1234567?\nAnswer: 24...   
1  6348700  4041201  Question: What is 1234567+1234567?\nAnswer: 24...   
2  4524571  9012469  Question: What is 1234567+1234567?\nAnswer: 24...   
3  3044419  6608684  Question: What is 1234567+1234567?\nAnswer: 24...   
4  1756139  8493797  Question: What is 1234567+1234567?\nAnswer: 24...   
5  8083884  3154325  Question: What is 1234567+1234567?\nAnswer: 24...   
6  8888358  1527113  Question: What is 1234567+1234567?\nAnswer: 24...   
7  4025054  2352516  Question: What is 1234567+1234567?\nAnswer: 24...   
8  5053054  8166918  Question: What is 1234567+1234567?\nAnswer: 24...   
9  3075780  1468192  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10040593  10034543    False  
1  10389901  10389301    False  
2  13537040   5436000    False  
3   9653103   9653093    False  
4  10249936  1025817

-----------

**Q4d.** Let's look at a specific example with large absolute error.
* Run the cell at least 5 times. Does the error change each time? Why?
* Can you think of a prompt to reduce the error?
* Why do you think it would work?
* Does it work in practice? Why or why not?

**Answer:**

* **Run the cell at least 5 times. Does the error change with each time? Why?**
Yes, the error and the final output change across runs. Because the configuration uses a non-zero temperature (0.7) with sampling (top_k=50, top_p=0.6), the model samples tokens from a probability distribution rather than using deterministic decoding. The most frequent output (1111000) represents the model's most probable path, but at points of token uncertainty, random sampling selects less likely tokens, leading to varying errors.
* **Can you think of a prompt to reduce the error?**
Yes, a Chain-of-Thought (CoT) or step-by-step reasoning prompt can be added to guide the model:
"Let's solve this addition problem step by step, aligning the numbers by their place values from right to left to ensure accuracy:"
* **Why do you think it would work?**
Large integers like 9090909 and 1010101 are split by the tokenizer into sub-word tokens, hiding clear place-value alignment. A step-by-step prompt forces the model to allocate intermediate tokens to the problem, allowing its attention mechanism to break down the addition and carry operations incrementally instead of trying to predict a complex multi-digit result all at once.
* **Does it work in practice? Why or why not?**
No, it still fails to yield the exact correct sum (10101010) consistently. Even with structured prompting, language models lack a built-in algorithmic calculator or exact carry mechanism. Tokenization boundaries and limitations in internal representations mean the model still makes compounding arithmetic errors, proving that prompt engineering alone is insufficient for precise large-integer math.

In [23]:
test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    fixed_pairs=[(9090909,1010101)],
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    model_obj=model,
    tokenizer_obj=tokenizer,
    debug=True
)

  0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
1111000
Question: What is 5555555+2222222?
Answer: 7777777
Question: What is 3333

************ Final Output ************
1111000
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010   1111000    False  


{'res': np.float64(-0.0),
 'acc': np.float64(0.0),
 'mae': 8990010.0,
 'prompt_length': 79}

In [24]:
test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    fixed_pairs=[(9090909,1010101)],
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    model_obj=model,
    tokenizer_obj=tokenizer,
    debug=True
)

  0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
1111000
Question: What is 3456789+2345678?
Answer: 5792467
Question: What is 4567

************ Final Output ************
1111000
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010   1111000    False  


{'res': np.float64(-0.0),
 'acc': np.float64(0.0),
 'mae': 8990010.0,
 'prompt_length': 79}

In [25]:
test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    fixed_pairs=[(9090909,1010101)],
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    model_obj=model,
    tokenizer_obj=tokenizer,
    debug=True
)

  0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
10100909
Question: What is 5555555+2222222?
Answer: 7777777
Question: What is 333

************ Final Output ************
10100909
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010  10100909    False  


{'res': np.float64(0.0),
 'acc': np.float64(0.0),
 'mae': 101.0,
 'prompt_length': 79}

In [26]:
test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    fixed_pairs=[(9090909,1010101)],
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    model_obj=model,
    tokenizer_obj=tokenizer,
    debug=True
)

  0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
1111000
Question: What is 5555555+2222222?
Answer: 7777777
Question: What is 3333

************ Final Output ************
1111000
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10101010   1111000    False  


{'res': np.float64(-0.0),
 'acc': np.float64(0.0),
 'mae': 8990010.0,
 'prompt_length': 79}

In [27]:
test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    fixed_pairs=[(9090909,1010101)],
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    model_obj=model,
    tokenizer_obj=tokenizer,
    debug=True
)

  0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


************ Prompt ************
Question: What is 1234567+1234567?
Answer: 2469134
Question: What is 9090909+1010101?
Answer: 

************ Raw Response ************
101010091
Question: What is 1111111+2222222?
Answer: 3333333
Question: What is 33

************ Final Output ************
101010091
     int_a    int_b                                             prompt  \
0  9090909  1010101  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer   response  correct  
0  10101010  101010091    False  


{'res': np.float64(-0.0),
 'acc': np.float64(0.0),
 'mae': 90909081.0,
 'prompt_length': 79}

**Q4d experiment: step-by-step prompt on 9090909+1010101.** Run in a separate Colab session because of GPU limits (`tok` = Llama-2-7b-chat tokenizer, `model` = Llama-2-7b-chat).

In [ ]:
added_prompt = "Let's solve this addition problem step by step, aligning the numbers by their place values from right to left to ensure accuracy:"
suffix = "\nAnswer: "
prompt = added_prompt + "9090909+1010101" + suffix
cfg = dict(max_new_tokens=100, temperature=0.7, top_k=50, top_p=0.6, repetition_penalty=1.0, do_sample=True)

for run in range(5):
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, **cfg, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    first = re.search(r"\d{1,}", text.split("\n")[0])
    pred = int(first.group()) if first else 0
    print(f"run {run+1}: {pred}  (correct 10101010, error {abs(10101010 - pred):,})")

run 1: 101101090  (correct 10101010, error 91,000,080)
